In [0]:
spark.sql("SHOW CATALOGS").show()


In [0]:
spark.sql("show schemas in train").show()

In [0]:
spark.sql("show tables in train.train_schema").show()

In [0]:
dbutils.fs.ls("/Volumes/train/train_schema/train_data/")


In [0]:
df = spark.read \
    .option("header","true") \
    .option("inferSchema","true") \
    .csv("/Volumes/train/train_schema/train_data/train.csv")
df.show(5)

In [0]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("train.train_schema.train_row")

In [0]:
df.show(6)

In [0]:
display(df)

In [0]:
df.columns

In [0]:
df_clean =df
for c in df.columns:
    new_c = (
        c.lower()
        .replace(" ","_")
        .replace("-","_")
    )
    df_clean = df_clean.withColumnRenamed(c,new_c)

In [0]:
df_clean.columns

In [0]:
df_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("train.train_Schema.train_row")    

In [0]:
spark.sql("SHOW TABLES in train.train_schema").show(truncate=False)

In [0]:
df_clean.show(5)

In [0]:
df_clean.printSchema()
display(df_clean)

In [0]:
df_clean.select("order_id","customer_id").show(5)

In [0]:
from pyspark.sql.functions import col
df_num = df_clean \
    .withColumn("sales",col("sales").cast("double")) \
    .withColumn("quantity",col("quantity").cast("int")) \
    .withColumn("discount",col("discount").cast("double"))

In [0]:
df_num.printSchema()

In [0]:
from pyspark.sql.functions import col
df_clean.filter(~col("sales").rlike("^[0-9.]+$")).show()

In [0]:
from pyspark.sql.functions import regexp_replace

df_fixed = df_clean.withColumn(
    "sales",
    regexp_replace(col("sales"), "[^0-9.]", "").cast("double")
)


In [0]:
df_fixed.printSchema()

In [0]:
df_fixed2 = df_fixed.withColumn(
"quantity",regexp_replace(col("quantity"),"[^0-9]","").cast("int")
)

In [0]:
df_fixed2 = df_fixed2.withColumn("discount",
                                 regexp_replace(col("discount"),"[^0-9.]","").cast("double"))

In [0]:
df_fixed2.printSchema()

In [0]:
spark.sql("show tables in train.train_schema").show()

In [0]:
df=df_fixed2


In [0]:
from pyspark.sql.functions import col, when

df = df.withColumn(
    "sales",
    when(col("sales") == "", None).otherwise(col("sales"))
)

df = df.withColumn("sales", col("sales").cast("double"))


In [0]:
from pyspark.sql.functions import col, regexp_replace, when, trim

df = df.withColumn(
    "sales",
    regexp_replace(col("sales"), "[^0-9.]", "")
)

df = df.withColumn(
    "sales",
    when(trim(col("sales")) == "", None).otherwise(col("sales"))
)


In [0]:
df = df.withColumn("sales", col("sales").cast("double"))


In [0]:
from pyspark.sql.functions import expr

df.groupBy("category") \
  .agg(expr("sum(try_cast(sales as double))").alias("total_sales")) \
  .show()


In [0]:
from pyspark.sql.functions import sum
df.groupBy("category") \
    .agg(
      sum("quantity").alias("total_quantity"),
      avg("quantity").alias("avg_quantity")
  ) \
  .show()


In [0]:
df.select("sales").show()

In [0]:
df.select(expr("try_cast(sales as double)").alias("sales")).show()


In [0]:
# 2. Print the schema of the DataFrame.
df.printSchema()

In [0]:
# Q3. Display the first 10 rows of the dataset
df.show(10)

In [0]:
# Count the total number of records (rows) present in the dataset
df.count()

In [0]:
# . Count the total number of colen
len(df.columns)

In [0]:
df.select("Orders","Customers","Products","Shipping","Sales_metrics").show()